[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI4EPS/DAS_Seismology_Workshop/blob/main/notebooks/lab2_phasenet_das/Notebooks/lab2d_dasnet_inference.ipynb)

# Lab 2d: DASNet Inference

In this lab, you will learn how to:
- Load one Monterey Bay DAS tile from Hugging Face (**AI4EPS/quakeflow_das**), same flow as Lab 2a;
- Run DASNet for detection, classification, masks, and arrival-oriented picks;
- Inspect saved JSON and quicklook figures;
- Optionally run the same pipeline through predict.py from the shell.

**References:**
- Zhang, C., et al. (2026). "A deep learning framework for marine acoustic and seismic monitoring with distributed acoustic sensing." [arXiv:2603.14844](https://arxiv.org/abs/2603.14844).
- Romanowicz, B., et al. (2023). "SeaFOAM: A year‐long DAS deployment in Monterey Bay, California." *SRL*, 94(5), 2348-2359.

---

## Setup


In [ ]:
!pip install dasnet huggingface_hub -q

In [ ]:
import json
import os
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import urllib.request
from IPython.display import Image, display
from huggingface_hub import hf_hub_download

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

DS = 2  # downsample factor for plotting


def plot_das(ax, arr, nx, nt, dt_s, **kwargs):
    """Plot a downsampled DAS 2D array."""
    arr = np.asarray(arr)
    ax.imshow(
        arr[::DS, ::DS],
        aspect="auto",
        interpolation="bilinear",
        extent=[0, nt * dt_s, nx, 0],
        **kwargs,
    )

## 1. Load DAS Data

We use DAS data from **SeaFOAM** in Monterey Bay, California (Romanowicz et al., 2023). **DASNet** was trained on SeaFOAM. The deployment uses a ~52 km fiber with 5.2 m channel spacing at 200 Hz sampling.

Shallow parts of the cable are dominated by cultural and sea-surface noise, so we focus on the **deep-water** section (channel index > 7400).

Each workshop tile is an HDF5 file containing a 2D array of shape (nx, nt) — channels by time — plus metadata for the sample interval.

Set **EVENT_ID** in the next cell. The following figure shows **raw strain rate**, **2–10 Hz bandpass**, and **>10 Hz highpass**.


In [ ]:
HF_REPO = "AI4EPS/quakeflow_das"
EVENT_ID = "20231109T132510Z"

h5_path = hf_hub_download(
    HF_REPO,
    f"monterey_bay/data/{EVENT_ID}.h5",
    repo_type="dataset",
    local_dir="data/quakeflow_das",
)
SELECTED_H5 = str(Path(h5_path).resolve())

with h5py.File(SELECTED_H5, "r") as fp:
    data = fp["data"][:]
    attrs = dict(fp["data"].attrs)

dt_s = float(attrs.get("dt_s"))
begin_time = str(attrs.get("begin_time", ""))
nx, nt = data.shape

print(f"Event: {EVENT_ID}")
print(f"Shape: (nx={nx}, nt={nt})")
print(f"Sampling interval: {dt_s} s ({1/dt_s:.0f} Hz)")
print(f"Duration: {nt * dt_s:.1f} s")
if begin_time:
    print(f"Begin time: {begin_time}")

In [ ]:
from scipy.signal import sosfiltfilt
from dasnet.data.das import _safe_design_sos_bandpass, _safe_design_sos_highpass

# DASNet preprocesses strain rate into 3 channels:
#   channel 0 = raw strain rate (z-score normalized)
#   channel 1 = 2–10 Hz bandpass (z-score normalized)
#   channel 2 = >10 Hz highpass  (z-score normalized)
sos_bp = _safe_design_sos_bandpass(dt_s, 2.0, 10.0, order=4)
sos_hp = _safe_design_sos_highpass(dt_s, 10.0, order=4)
das_bp = sosfiltfilt(sos_bp, data, axis=1).astype(np.float32)
das_hp = sosfiltfilt(sos_hp, data, axis=1).astype(np.float32)

fig, axes = plt.subplots(3, 1, figsize=(12, 8))

titles = ["Raw strain rate", "2–10 Hz bandpass", ">10 Hz highpass"]
arrays = [data, das_bp, das_hp]

for ax, arr, title in zip(axes, arrays, titles):
    vmax = 2 * float(np.percentile(np.abs(arr), 95))
    plot_das(ax, arr, nx, nt, dt_s, cmap="seismic", vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Channel index")

fig.suptitle(f"DAS waveform — {EVENT_ID}", y=1.02)
plt.tight_layout()
plt.show()

## 2. Run DASNet

DASNet outputs a class, confidence, box, and soft mask per detection; peaks along the mask summarize arrivals where applicable. The next cell downloads the workshop checkpoint, runs the **selected** tile (**EVENT_ID** above) through the model, and saves JSON plus a figure under pred_notebook/.


In [ ]:
from dasnet import (
    build_dasnet_model,
    default_device,
    extract_peaks_for_instances,
    filter_by_score,
    forward_raw,
    load_checkpoint,
    make_infer_dataloader,
    plot_das_predictions,
    postprocess_batch,
    save_predictions_json,
)

# Download checkpoint
CKPT_URL = "https://github.com/czh4ng/DASNet-workshop/releases/download/v0.1.1/checkpoint.pth"
CKPT_PATH = Path("checkpoint.pth")
if not CKPT_PATH.exists():
    urllib.request.urlretrieve(CKPT_URL, CKPT_PATH)

# Build model and run inference
RESIZE_SCALE = 0.5
MIN_PROB = 0.8

device = default_device()
model = build_dasnet_model()
load_checkpoint(model, str(CKPT_PATH), device)

loader, _ = make_infer_dataloader(
    [SELECTED_H5],
    batch_size=1,
    num_workers=0,
    resize_scale=RESIZE_SCALE,
    storage_backend="local",
)
images, names = next(iter(loader))

raw_outputs = forward_raw(model, list(images), device)
_, processed = postprocess_batch(names, raw_outputs)
selected = filter_by_score(processed[0], MIN_PROB)
peak_points_list, peak_scores_list = extract_peaks_for_instances(selected)

# Save results
RESULT_DIR = Path("pred_notebook")
FIG_DIR = RESULT_DIR / "figures_dasnet"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

fn0 = names[0]
np_selected = {k: (v.detach().cpu().numpy() if torch.is_tensor(v) else v) for k, v in selected.items()}
save_predictions_json(fn0, np_selected, peak_points_list, peak_scores_list, str(RESULT_DIR), resize_scale=RESIZE_SCALE)

fig_path = FIG_DIR / (Path(fn0).stem + ".jpg")
plot_das_predictions(images[0].cpu().numpy(), np_selected, str(fig_path), score_threshold=MIN_PROB)

print(f"Detected {len(selected['scores'])} instances → {RESULT_DIR}")

## 3. Check output

Open the quicklook figure and JSON for the **same** tile as **EVENT_ID** / Run DASNet. Each instance stores the box and (when present) pick coordinates in **full-resolution** channel/time units after undoing the resize, together with class id, score, and per-pick mask scores.


In [ ]:
if fig_path.exists():
    display(Image(filename=str(fig_path)))

# Show JSON (truncate long pick lists for readability)
json_path = RESULT_DIR / (Path(fn0).stem + ".json")
payload = json.loads(json_path.read_text())
for inst in payload.get("instances", []):
    for key in ("picks", "pick_scores"):
        if key in inst and len(inst[key]) > 5:
            inst[key] = inst[key][:5] + [f"... ({len(inst[key])} total)"]

print(json.dumps(payload, indent=2))

## Optional: Batch inference

Run the same pipeline on multiple events. Each tile is downloaded from Hugging Face, processed through DASNet, and saved as JSON + quicklook figure.

In [ ]:
from tqdm.auto import tqdm

BATCH_EVENT_IDS = [
    "20231116T130210Z",
    "20231110T105410Z",
    "20231124T104510Z",
    "20231130T194610Z",
    "20240204T004410Z",
    "20240428T100822Z",
    "20220828T062358Z",
    "20220828T140758Z",
    "20230407T074209Z",
]

# Download all tiles
h5_paths = []
for eid in tqdm(BATCH_EVENT_IDS, desc="Downloading", unit="file"):
    p = hf_hub_download(
        "AI4EPS/quakeflow_das",
        f"monterey_bay/data/{eid}.h5",
        repo_type="dataset",
        local_dir="data/quakeflow_das",
    )
    h5_paths.append(str(Path(p).resolve()))

# Batch inference
BATCH_DIR = Path("pred_batch")
BATCH_FIG_DIR = BATCH_DIR / "figures_dasnet"
BATCH_DIR.mkdir(parents=True, exist_ok=True)
BATCH_FIG_DIR.mkdir(parents=True, exist_ok=True)

loader, _ = make_infer_dataloader(h5_paths, batch_size=1, num_workers=0, resize_scale=RESIZE_SCALE, storage_backend="local")

for images, names in tqdm(loader, desc="Predicting", unit="file"):
    raw_out = forward_raw(model, list(images), device)
    fnames, results = postprocess_batch(names, raw_out)
    for i, fn in enumerate(fnames):
        sel = filter_by_score(results[i], MIN_PROB)
        if len(sel["scores"]) == 0:
            continue
        pts, scores = extract_peaks_for_instances(sel)
        np_sel = {k: (v.detach().cpu().numpy() if torch.is_tensor(v) else v) for k, v in sel.items()}
        save_predictions_json(fn, np_sel, pts, scores, str(BATCH_DIR), resize_scale=RESIZE_SCALE)
        plot_das_predictions(images[i].cpu().numpy(), np_sel, str(BATCH_FIG_DIR / (Path(fn).stem + ".jpg")), score_threshold=MIN_PROB)

n_json = len(list(BATCH_DIR.glob("*.json")))
print(f"Done. {n_json} JSON files saved to {BATCH_DIR}")

In [ ]:
FIG_PREVIEW_N = None  # None = show all; set int to limit

figs = sorted(BATCH_FIG_DIR.glob("*.jpg"))
if FIG_PREVIEW_N is not None:
    figs = figs[:FIG_PREVIEW_N]

print(f"Showing {len(figs)} of {len(list(BATCH_FIG_DIR.glob('*.jpg')))} figures")
for fig in figs:
    print(fig.stem)
    display(Image(filename=str(fig)))
    print()